# Llama-3.1-8B-Instruct in 4-bit bitsandbytes

Let's first install the required libraries:

In [ ]:
%pip install transformers[torch] bitsandbytes

Note: you may need to restart the kernel to use updated packages.


We import the required libraries : 

In [1]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install transformers

Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig  

C:\Users\victo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import bitsandbytes
print("bitsandbytes importé avec succès !")


bitsandbytes importé avec succès !


In [5]:
import torch
print(torch.cuda.is_available())  # Doit retourner True si le GPU est détecté


True


Let's load the model. To quantize the model on the fly, we pass a `quantization_config`:

In [6]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "Llama-3.2-3B-Instruct"

# Define the quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Load the model explicitly in PyTorch ('pt') framework
quantized_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",  # Automatically choose the device (CPU or GPU)
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    framework="pt"  # Explicitly use PyTorch
)


RuntimeError: Failed to import transformers.models.llama.modeling_llama because of the following error (look up to see its traceback):
Traceback (most recent call last):
  File "C:\Users\victo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tensorflow\python\pywrap_tensorflow.py", line 73, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: Une routine d’initialisation d’une bibliothèque de liens dynamiques (DLL) a échoué.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

Then, we need to prepare the inputs: 

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
input_text = "What are we having for dinner?"
input_ids = tokenizer(input_text, return_tensors="pt").to("cuda")

Finally, we can generate the output ! 

In [ ]:
output = quantized_model.generate(**input_ids, max_new_tokens=10)

print(tokenizer.decode(output[0], skip_special_tokens=True))